# Momentum Engineering (Notebook)

This notebook mirrors `momentum.py` and adds visual checks at each step.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
BASE_DIR = Path(".")
valid_bookings_df = pd.read_parquet(BASE_DIR / "valid_bookings.parquet")
valid_bookings_df.head()

## 0) Guardrails / required cols

In [ ]:
required_cols = ["restaurant_id", "booking_date", "id", "revenue_dollars"]
missing = [c for c in required_cols if c not in valid_bookings_df.columns]
if missing:
    raise ValueError(f"valid_bookings_df missing required columns: {missing}")

valid_bookings_df["booking_date"] = pd.to_datetime(valid_bookings_df["booking_date"], errors="coerce")
valid_bookings_df = valid_bookings_df.dropna(subset=["booking_date", "restaurant_id"])

if "total_guests" not in valid_bookings_df.columns:
    if "adult" in valid_bookings_df.columns and "kids" in valid_bookings_df.columns:
        valid_bookings_df["total_guests"] = (
            valid_bookings_df["adult"].fillna(0) + valid_bookings_df["kids"].fillna(0)
        )
    else:
        valid_bookings_df["total_guests"] = np.nan

valid_bookings_df.shape

## 1) Monthly aggregation

In [ ]:
valid_bookings_df["year_month"] = valid_bookings_df["booking_date"].dt.to_period("M").dt.to_timestamp()

# Filter out advance bookings beyond Dec 2025
cutoff_month = pd.Timestamp("2025-12-01")
valid_bookings_df = valid_bookings_df[valid_bookings_df["year_month"] <= cutoff_month].copy()

restaurants_agg = (
    valid_bookings_df
    .groupby(["restaurant_id", "year_month"], as_index=False)
    .agg(
        monthly_bookings=("id", "count"),
        monthly_revenue=("revenue_dollars", "sum"),
        avg_revenue_per_booking=("revenue_dollars", "mean"),
        avg_guests=("total_guests", "mean"),
        active_days=("booking_date", lambda x: x.dt.date.nunique()),
    )
)

restaurants_agg["monthly_bookings"] = restaurants_agg["monthly_bookings"].fillna(0).astype(int)
restaurants_agg["monthly_revenue"] = restaurants_agg["monthly_revenue"].fillna(0.0)
restaurants_agg["avg_revenue_per_booking"] = restaurants_agg["avg_revenue_per_booking"].replace([np.inf, -np.inf], np.nan)
restaurants_agg["avg_revenue_per_booking"] = restaurants_agg["avg_revenue_per_booking"].fillna(0.0)

restaurants_agg = restaurants_agg.sort_values(["restaurant_id", "year_month"]).reset_index(drop=True)
restaurants_agg.head()

In [ ]:
# Plot bookings and revenue distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(restaurants_agg["monthly_bookings"], bins=30, ax=axes[0])
axes[0].set_title("Monthly bookings distribution")

sns.histplot(restaurants_agg["monthly_revenue"], bins=30, ax=axes[1])
axes[1].set_title("Monthly revenue distribution")
plt.tight_layout()

## 2) Minimum history filter

In [ ]:
MIN_MONTHS = 3
hist = restaurants_agg.groupby("restaurant_id")["year_month"].nunique()
keep_ids = hist[hist >= MIN_MONTHS].index
restaurants_agg = restaurants_agg[restaurants_agg["restaurant_id"].isin(keep_ids)].copy()
restaurants_agg.shape

## 3) Winsorise extremes

In [ ]:
def winsorise_series(s: pd.Series, lower_q=0.01, upper_q=0.99) -> pd.Series:
    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)
    return s.clip(lower=lo, upper=hi)

for col in ["monthly_bookings", "monthly_revenue", "avg_revenue_per_booking"]:
    restaurants_agg[col] = winsorise_series(restaurants_agg[col])

restaurants_agg[["monthly_bookings","monthly_revenue","avg_revenue_per_booking"]].describe().T

## 4) Growth + rolling growth

In [ ]:
restaurants_agg["booking_growth_pct"] = (
    restaurants_agg.groupby("restaurant_id")["monthly_bookings"]
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
)

restaurants_agg["revenue_growth_pct"] = (
    restaurants_agg.groupby("restaurant_id")["monthly_revenue"]
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
)

ROLL = 3
restaurants_agg["booking_growth_rolling"] = (
    restaurants_agg.groupby("restaurant_id")["booking_growth_pct"]
    .rolling(ROLL, min_periods=ROLL)
    .mean()
    .reset_index(level=0, drop=True)
)

restaurants_agg["revenue_growth_rolling"] = (
    restaurants_agg.groupby("restaurant_id")["revenue_growth_pct"]
    .rolling(ROLL, min_periods=ROLL)
    .mean()
    .reset_index(level=0, drop=True)
)

restaurants_agg["booking_growth_rolling"] = restaurants_agg["booking_growth_rolling"].fillna(0.0)
restaurants_agg["revenue_growth_rolling"] = restaurants_agg["revenue_growth_rolling"].fillna(0.0)

restaurants_agg[["booking_growth_rolling","revenue_growth_rolling"]].describe().T

In [ ]:
# Growth distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(restaurants_agg["booking_growth_rolling"], bins=30, ax=axes[0])
axes[0].set_title("Rolling booking growth (3m)")

sns.histplot(restaurants_agg["revenue_growth_rolling"], bins=30, ax=axes[1])
axes[1].set_title("Rolling revenue growth (3m)")
plt.tight_layout()

## 5) Standardize into comparable scores

In [ ]:
def pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

restaurants_agg["perf_bookings_rank"] = pct_rank(restaurants_agg["monthly_bookings"])
restaurants_agg["perf_spend_rank"] = pct_rank(restaurants_agg["avg_revenue_per_booking"])
restaurants_agg["performance_score"] = (restaurants_agg["perf_bookings_rank"] + restaurants_agg["perf_spend_rank"]) / 2

restaurants_agg["growth_bookings_rank"] = pct_rank(restaurants_agg["booking_growth_rolling"])
restaurants_agg["growth_revenue_rank"] = pct_rank(restaurants_agg["revenue_growth_rolling"])
restaurants_agg["growth_score"] = (restaurants_agg["growth_bookings_rank"] + restaurants_agg["growth_revenue_rank"]) / 2

restaurants_agg[["performance_score","growth_score"]].describe().T

In [ ]:
# Performance vs Growth scatter
plt.figure(figsize=(6, 5))
sns.scatterplot(
    data=restaurants_agg,
    x="performance_score",
    y="growth_score",
    alpha=0.4,
    s=20
)
plt.title("Performance vs Growth")
plt.tight_layout()

## 6) Composite momentum + segmentation

In [ ]:
ALPHA = 0.5
restaurants_agg["momentum_score"] = ALPHA * restaurants_agg["performance_score"] + (1 - ALPHA) * restaurants_agg["growth_score"]

perf_cut = restaurants_agg["performance_score"].median()
grow_cut = restaurants_agg["growth_score"].median()

restaurants_agg["momentum_segment"] = np.select(
    [
        (restaurants_agg["performance_score"] >= perf_cut) & (restaurants_agg["growth_score"] >= grow_cut),
        (restaurants_agg["performance_score"] <  perf_cut) & (restaurants_agg["growth_score"] >= grow_cut),
        (restaurants_agg["performance_score"] >= perf_cut) & (restaurants_agg["growth_score"] <  grow_cut),
        (restaurants_agg["performance_score"] <  perf_cut) & (restaurants_agg["growth_score"] <  grow_cut),
    ],
    [
        "Rising Stars",
        "Emerging Opportunities",
        "Established Players",
        "Needs Attention",
    ],
    default="Unclassified"
)

restaurants_agg[["momentum_score","momentum_segment"]].head()

In [ ]:
# Segment counts
plt.figure(figsize=(6, 4))
segment_counts = restaurants_agg["momentum_segment"].value_counts()
sns.barplot(x=segment_counts.index, y=segment_counts.values)
plt.title("Momentum segment counts")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

## 7) Latest-month prioritised list

In [ ]:
latest_month = restaurants_agg["year_month"].max()
priority_latest = (
    restaurants_agg[restaurants_agg["year_month"] == latest_month]
    .sort_values("momentum_score", ascending=False)
    .reset_index(drop=True)
)

print("Latest month:", latest_month.date())
print(priority_latest["momentum_segment"].value_counts(dropna=False))
priority_latest.head(10)

In [ ]:
# Save latest-month list
output_path = BASE_DIR / "priority_latest_momentum_labels.parquet"
priority_latest.to_parquet(output_path, index=False)
print("Saved latest-month momentum labels to:", output_path)